In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# 自动向上查找项目根目录 (含 .gitignore 的文件夹)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# 切换 cwd 到项目根, 使所有相对路径 (Stage1_Exploration/, Refined_Results_v4/ 等) 保持有效
os.chdir(PROJECT_ROOT)
# 让 notebooks 能 `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from pysr import PySRRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

# ==========================================
# 0. 实验参数与路径配置 (Configuration)
# ==========================================
# 定义一系列噪声水平 (0% - 200%)，用于全面评估模型的鲁棒性
NOISE_LEVELS = [0,0.1,0.2,0.3,0.35,0.40,0.45,0.50,0.55,0.6,0.65,0.7,0.75,0.8,0.9,1.0,1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9,2.0]

# 结果保存目录 (Refined_Results_v5)
BASE_DIR = "Refined_Results_v5" 
if not os.path.exists(BASE_DIR):
    os.makedirs(BASE_DIR)

FEATURE_NAMES = ['V_in', 'C_in', 'Area', 'Distance']
SCALE = 1e7 # 浓度缩放因子

# 用于收集所有噪声水平下的 R2 结果，最终汇总为 CSV
summary_list = []

# ==========================================
# 1. 数据加载与预处理 (Data Loading & Preprocessing)
# ==========================================
print("正在加载并按 Case 划分数据...")
df = pd.read_csv('data/train_dataset_ready.csv')

# 浓度预缩放
df['C_in'] = df['C_in'] * SCALE
df['C_out'] = df['C_out'] * SCALE

# 按 Case 划分训练集和测试集
# 保证测试集中的工况 (Case) 是模型从未见过的
unique_cases = df['Case'].unique()
train_cases, test_cases = train_test_split(unique_cases, test_size=0.3, random_state=42)

# 筛选数据
train_df = df[df['Case'].isin(train_cases)].copy()
test_df = df[df['Case'].isin(test_cases)].copy()

print(f"划分完成: 训练集 Case 数: {len(train_cases)}, 测试集 Case 数: {len(test_cases)}")
print(f"训练集行数: {len(train_df)}, 测试集行数: {len(test_df)}")

# 准备训练数据 (Clean Baseline)
# 注意：X_train_clean 和 y_train_clean 是干净的基准数据
# 在后续循环中，我们会人为向 y_train_clean 添加噪声
X_train_clean = train_df[FEATURE_NAMES].values
y_train_clean = train_df['C_out'].values

# 准备测试数据 (Ground Truth)
# 测试集始终保持干净，用于评估模型在真实场景下的准确度
X_test = test_df[FEATURE_NAMES].values
y_test_clean = test_df['C_out'].values

# 保留测试集的辅助列 (Case ID 和 Distance)，用于后续按 Case 画图分析
test_cases_col = test_df['Case'].values
test_dist_col = test_df['Distance'].values

# ==========================================
# 2. 自动化实验循环 (Automated Experiment Loop)
# ==========================================
# 遍历每一个定义的噪声水平，对比 Direct PySR, MLP 和 Hybrid PySR 的表现
for noise_pct in NOISE_LEVELS:
    noise_label = f"{int(noise_pct*100)}pct"
    path = os.path.join(BASE_DIR, f"Noise_{noise_label}")
    if not os.path.exists(path): os.makedirs(path)
    
    print(f"\n>>> 运行实验: 噪声等级 {noise_pct*100}%")

    # --- A. 构造带噪训练集 (Noise Injection) ---
    # 仅对训练集的目标值 y 添加高斯白噪声，模拟传感器误差或环境干扰
    np.random.seed(42)
    noise_train = np.random.normal(0, noise_pct * y_train_clean) if noise_pct > 0 else 0
    y_train_noisy = y_train_clean + noise_train
    
    # 构造带噪测试集 (仅用于观察 MLP 对噪声数据的拟合能力，不作为最终指标)
    noise_test = np.random.normal(0, noise_pct * y_test_clean) if noise_pct > 0 else 0
    y_test_noisy = y_test_clean + noise_test
    
    # 物理约束：浓度不能为负，设置下限为 1e-6
    y_train_noisy = np.maximum(y_train_noisy, 1e-6)
    y_test_noisy = np.maximum(y_test_noisy, 1e-6)

    # --- B. 任务 1: 直接符号回归 (Direct PySR Baseline) ---
    # 直接使用带噪数据训练 PySR，作为对照组 (Baseline)
    print(f"    [Task 1] 运行直接 PySR...")
    # 随机抽样 5000 个点加速训练
    pysr_sub_idx = np.random.choice(len(y_train_noisy), 5000, replace=False)
    
    model_direct = PySRRegressor(
        niterations=150,
        populations=30,
        maxsize=25,
        binary_operators=["+", "*", "-", "/"],
        unary_operators=["exp", "log", "sqrt", "square", "inv(x)=1/x"],
        extra_sympy_mappings={'inv': lambda x: 1/x},
        temp_equation_file=True,
        delete_tempfiles=False, # 保留临时文件以备查
        verbosity=0
    )
    # 训练
    model_direct.fit(X_train_clean[pysr_sub_idx], y_train_noisy[pysr_sub_idx], variable_names=FEATURE_NAMES)
    # 保存公式到 CSV
    model_direct.equations_.to_csv(os.path.join(path, f"all_formulas_direct_{noise_label}.csv"))
    # 在干净测试集上预测
    y_pred_direct = model_direct.predict(X_test)

    # --- C. 任务 2: 神经网络训练 (MLP Training) ---
    # 训练 MLP 作为一个非线性去噪器
    print(f"    [Task 2] 训练 MLP...")
    # 数据标准化 (Standardization) 对 MLP 至关重要
    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train_clean)
    X_test_scaled = scaler_X.transform(X_test)

    mlp = MLPRegressor(
        hidden_layer_sizes=(128, 64, 32), # 3层隐藏层
        learning_rate_init=0.001,
        activation='relu',
        max_iter=1000,      
        random_state=42,
        early_stopping=False
    )
    mlp.fit(X_train_scaled, y_train_noisy)

    # 保存 Loss 曲线，用于检查收敛情况
    pd.DataFrame(mlp.loss_curve_, columns=['Loss']).to_csv(os.path.join(path, "mlp_loss_history.csv"))
    # 预测
    y_pred_mlp = mlp.predict(X_test_scaled)
    
    # --- D. 任务 3: 混合框架 (Hybrid Framework) ---
    # 策略：先用 MLP 对训练数据进行"去噪"，再用 PySR 拟合去噪后的数据
    print(f"    [Task 3] 运行混合 PySR...")
    
    # 1. MLP 去噪 (Denoising): 使用训练好的 MLP 预测训练集子集
    # 假设 MLP 学习到了潜在的物理流形，其预测值比原始含噪标签更接近真值
    y_denoised_train_sub = mlp.predict(X_train_scaled[pysr_sub_idx])
    
    # 2. PySR 拟合 (Symbolic Regression): 拟合去噪后的标签
    model_hybrid = PySRRegressor(
        niterations=150,
        populations=30,
        maxsize=25,
        binary_operators=["+", "*", "-", "/"],
        unary_operators=["exp", "log", "sqrt", "square", "inv(x)=1/x"],
        extra_sympy_mappings={'inv': lambda x: 1/x},
        temp_equation_file=True,
        delete_tempfiles=False,
        verbosity=0
    )
    model_hybrid.fit(X_train_clean[pysr_sub_idx], y_denoised_train_sub, variable_names=FEATURE_NAMES)
    # 保存公式
    model_hybrid.equations_.to_csv(os.path.join(path, f"all_formulas_hybrid_{noise_label}.csv"))
    # 预测
    y_pred_hybrid = model_hybrid.predict(X_test)

    # --- E. 结果保存 (Saving Predictions) ---
    # 保存所有方法的预测结果，用于后续绘制 Parity Plot 和 序列对比图
    print(f"    [Saving] 保存预测对比数据...")
    comparison_df = pd.DataFrame({
        'Case': test_cases_col,
        'Distance': test_dist_col,
        'True_Clean': y_test_clean,        # Ground Truth
        'True_Noisy': y_test_noisy,        # Noisy Measurement
        'Pred_Direct_PySR': y_pred_direct, # Baseline Prediction
        'Pred_MLP': y_pred_mlp,            # Intermediate Denoising
        'Pred_Hybrid_PySR': y_pred_hybrid  # Final Hybrid Prediction
    })
    comparison_df.to_csv(os.path.join(path, "predictions_comparison.csv"), index=False)

    # --- F. 指标计算与汇总 (Metric Calculation) ---
    r2_direct = r2_score(y_test_clean, y_pred_direct)
    r2_mlp = r2_score(y_test_clean, y_pred_mlp)
    r2_hybrid = r2_score(y_test_clean, y_pred_hybrid)
    
    summary_list.append({
        'Noise_Ratio': noise_pct,
        'R2_Direct_PySR': r2_direct,
        'R2_MLP': r2_mlp,
        'R2_Hybrid_PySR': r2_hybrid
    })

    print(f"    结果: Direct R2={r2_direct:.4f}, Hybrid R2={r2_hybrid:.4f}")
    
    # 保存训练好的模型对象
    joblib.dump(mlp, os.path.join(path, "mlp_model.pkl"))
    joblib.dump(scaler_X, os.path.join(path, "scaler_X.pkl"))

# ==========================================
# 3. 最终汇总 (Final Summary)
# ==========================================
# 保存不同噪声水平下的 R2 汇总表，用于绘制鲁棒性曲线 (Figure 6)
summary_df = pd.DataFrame(summary_list)
summary_df.to_csv(os.path.join(BASE_DIR, "final_summary_r2.csv"), index=False)

print("\n" + "="*50)
print("所有实验结束！")
print(f"1. 折线图数据: {os.path.join(BASE_DIR, 'final_summary_r2.csv')}")
print(f"2. 散点图数据: 在每个 Noise_xxx 文件夹下的 predictions_comparison.csv 中")
print("="*50)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
正在加载并按 Case 划分数据...
划分完成: 训练集 Case 数: 350, 测试集 Case 数: 150
训练集行数: 38850, 测试集行数: 16650

>>> 运行实验: 噪声等级 110.00000000000001%
    [Task 1] 运行直接 PySR...


D:\Users\A\anaconda3\envs\py310_llm\lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


    [Task 2] 训练 MLP...
    [Task 3] 运行混合 PySR...


D:\Users\A\anaconda3\envs\py310_llm\lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


    [Saving] 保存预测对比数据...
    结果: Direct R2=0.7358, Hybrid R2=0.9605

>>> 运行实验: 噪声等级 120.0%
    [Task 1] 运行直接 PySR...


...

    [Saving] 保存预测对比数据...
    结果: Direct R2=0.6241, Hybrid R2=0.5576

>>> 运行实验: 噪声等级 300%
    [Task 1] 运行直接 PySR...


D:\Users\A\anaconda3\envs\py310_llm\lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


    [Task 2] 训练 MLP...
    [Task 3] 运行混合 PySR...


D:\Users\A\anaconda3\envs\py310_llm\lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


    [Saving] 保存预测对比数据...
    结果: Direct R2=-0.1550, Hybrid R2=-0.0908

所有实验结束！
1. 折线图数据: Refined_Results_v5\final_summary_r2.csv
2. 散点图数据: 在每个 Noise_xxx 文件夹下的 predictions_comparison.csv 中
